# 🏡 Feature Engineering Notebook — Ames Housing Dataset

**Goal:** Create meaningful engineered features from raw housing data, demonstrate measurable model improvement, and identify which features drive value.

**Target variable:** `SalePrice`

---


## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
SEED = 42

# ── Load data ───────────────────────────────────────────────────────────────────
df = pd.read_csv("train.csv")
print(f"Shape: {df.shape}")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T.round(2)


In [ ]:
# Missing value summary
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})


## 2. Feature Analysis

We separate numeric and categorical columns, then examine their relationships with `SalePrice`.


In [ ]:
TARGET = 'SalePrice'

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in [TARGET, 'Id']]
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features : {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")


In [ ]:
# Correlation of numeric features with SalePrice
corr_with_target = df[numeric_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr_with_target]
corr_with_target.plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
ax.set_title('Numeric Feature Correlations with SalePrice', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Top 12 numeric correlations (heatmap)
top12 = corr_with_target.abs().nlargest(12).index.tolist()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df[top12 + [TARGET]].corr(), annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5, ax=ax)
ax.set_title('Top-12 Numeric Features — Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Categorical mean SalePrice
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
key_cats = ['Neighborhood', 'OverallQual', 'ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish']
for ax, col in zip(axes.flat, key_cats):
    order = df.groupby(col)[TARGET].median().sort_values().index
    sns.boxplot(data=df, x=col, y=TARGET, order=order, ax=ax, palette='Blues_d')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.suptitle('SalePrice Distribution by Key Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Key takeaways from Feature Analysis:**
- `OverallQual`, `GrLivArea`, `GarageCars`, `GarageArea`, `TotalBsmtSF` are the strongest raw numeric predictors.
- `Neighborhood`, `ExterQual`, `KitchenQual`, `BsmtQual` are highly informative categorical features.
- Several date columns (`YearBuilt`, `YearRemodAdd`, `GarageYrBlt`) represent time, which we can convert to *age* features.


## 3. Basic Preprocessing

In [ ]:
df_processed = df.drop(columns=['Id']).copy()

# ── Numeric imputation (median) ─────────────────────────────────────────────────
num_cols_missing = df_processed[numeric_cols].isnull().sum()
num_cols_missing = num_cols_missing[num_cols_missing > 0].index.tolist()
for col in num_cols_missing:
    df_processed[col].fillna(df_processed[col].median(), inplace=True)

# ── Categorical imputation (mode / 'None') ──────────────────────────────────────
# For columns where NA is meaningful (e.g. no garage, no basement), fill with 'None'
none_cols = ['Alley','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
             'FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond',
             'PoolQC','Fence','MiscFeature','MasVnrType']
for col in none_cols:
    if col in df_processed.columns:
        df_processed[col].fillna('None', inplace=True)

# Remaining categoricals → mode
for col in categorical_cols:
    if col in df_processed.columns and df_processed[col].isnull().sum() > 0:
        df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)

# ── Ordinal encoding for quality columns ────────────────────────────────────────
qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_cols = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC',
             'KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']
for col in qual_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].map(qual_map).fillna(0).astype(int)

# ── One-hot encode remaining categoricals ───────────────────────────────────────
remaining_cats = [c for c in categorical_cols if c in df_processed.columns and c not in qual_cols]
df_processed = pd.get_dummies(df_processed, columns=remaining_cats, drop_first=True)

print(f"Processed shape: {df_processed.shape}")
df_processed.head(3)


In [ ]:
X_raw = df_processed.drop(columns=[TARGET])
y = df_processed[TARGET]
print(f"X_raw shape: {X_raw.shape}")
print(f"y shape    : {y.shape}")


## 4. Feature Engineering (Core)

We create new features across three categories: **numerical transformations**, **flag/categorical features**, and **interaction features**.


In [ ]:
fe = df_processed.copy()   # working DataFrame that accumulates new features
CURRENT_YEAR = 2010        # approx median YrSold in this dataset

# ════════════════════════════════════════════════════════════════════
# A. NUMERICAL FEATURES — Age and ratio transformations
# ════════════════════════════════════════════════════════════════════

# House age at time of sale (newer = higher value)
fe['House_Age']         = fe['YrSold'] - df['YearBuilt'].fillna(df['YearBuilt'].median())

# Years since last remodel at time of sale
fe['Remod_Age']         = fe['YrSold'] - df['YearRemodAdd'].fillna(df['YearRemodAdd'].median())

# Was the house remodelled at all?
fe['Was_Remodeled']     = (df['YearRemodAdd'] > df['YearBuilt']).astype(int)

# Total bathrooms (half baths count 0.5)
fe['Total_Bathrooms']   = (df['FullBath'].fillna(0)
                           + 0.5 * df['HalfBath'].fillna(0)
                           + df['BsmtFullBath'].fillna(0)
                           + 0.5 * df['BsmtHalfBath'].fillna(0))

# Total finished area (above grade + finished basement)
fe['Total_Finished_SF'] = (df['GrLivArea'].fillna(0)
                           + df['BsmtFinSF1'].fillna(0)
                           + df['BsmtFinSF2'].fillna(0))

# Total porch/deck area
fe['Total_Porch_SF']    = (df['WoodDeckSF'].fillna(0)
                           + df['OpenPorchSF'].fillna(0)
                           + df['EnclosedPorch'].fillna(0)
                           + df['3SsnPorch'].fillna(0)
                           + df['ScreenPorch'].fillna(0))

# Living area per room
fe['Area_Per_Room']     = df['GrLivArea'] / (df['TotRmsAbvGrd'].replace(0, 1))

# Living area per bedroom (comfort indicator)
fe['Area_Per_Bedroom']  = df['GrLivArea'] / (df['BedroomAbvGr'].replace(0, 1))

print("Numerical features added: House_Age, Remod_Age, Was_Remodeled, "
      "Total_Bathrooms, Total_Finished_SF, Total_Porch_SF, Area_Per_Room, Area_Per_Bedroom")


In [ ]:
# ════════════════════════════════════════════════════════════════════
# B. FLAG / CATEGORICAL FEATURES
# ════════════════════════════════════════════════════════════════════

# High-quality home flag (OverallQual >= 8)
fe['High_Quality_Flag']  = (df['OverallQual'] >= 8).astype(int)

# Has pool
fe['Has_Pool']           = (df['PoolArea'] > 0).astype(int)

# Has garage
fe['Has_Garage']         = (df['GarageArea'].fillna(0) > 0).astype(int)

# Has fireplace
fe['Has_Fireplace']      = (df['Fireplaces'] > 0).astype(int)

# Luxury flag: large area AND high quality
luxury_mask = (df['GrLivArea'] > df['GrLivArea'].quantile(0.75)) & (df['OverallQual'] >= 8)
fe['Luxury_Home_Flag']   = luxury_mask.astype(int)

# Neighborhood tier (manually based on known median prices in Ames data)
premium_nbhds  = ['NridgHt', 'NoRidge', 'StoneBr', 'Timber', 'Veenker', 'Somerst']
mid_nbhds      = ['CollgCr', 'Crawfor', 'ClearCr', 'Gilbert', 'NWAmes', 'Blmngtn', 'SawyerW']
fe['Neighborhood_Tier'] = df['Neighborhood'].apply(
    lambda n: 2 if n in premium_nbhds else (1 if n in mid_nbhds else 0)
)

print("Flag features added: High_Quality_Flag, Has_Pool, Has_Garage, "
      "Has_Fireplace, Luxury_Home_Flag, Neighborhood_Tier")


In [ ]:
# ════════════════════════════════════════════════════════════════════
# C. INTERACTION FEATURES
# ════════════════════════════════════════════════════════════════════

# Quality × Area interaction — premium homes get a much larger size premium
fe['Quality_x_Area']     = df['OverallQual'] * df['GrLivArea']

# Garage size × quality
fe['Garage_Score']       = df['GarageCars'].fillna(0) * df['GarageArea'].fillna(0)

# Basement quality × size
fe['Bsmt_Score']         = fe['BsmtQual'] * df['TotalBsmtSF'].fillna(0)

# Bedrooms × bathrooms (layout completeness)
fe['Bed_Bath_Interaction'] = df['BedroomAbvGr'] * fe['Total_Bathrooms']

# Overall quality squared (non-linear quality effect)
fe['OverallQual_Sq']     = df['OverallQual'] ** 2

# Log of GrLivArea (stabilises variance)
fe['Log_GrLivArea']      = np.log1p(df['GrLivArea'])

# Log of LotArea
fe['Log_LotArea']        = np.log1p(df['LotArea'])

print("Interaction features added: Quality_x_Area, Garage_Score, Bsmt_Score, "
      "Bed_Bath_Interaction, OverallQual_Sq, Log_GrLivArea, Log_LotArea")

new_features = ['House_Age','Remod_Age','Was_Remodeled','Total_Bathrooms',
                'Total_Finished_SF','Total_Porch_SF','Area_Per_Room','Area_Per_Bedroom',
                'High_Quality_Flag','Has_Pool','Has_Garage','Has_Fireplace',
                'Luxury_Home_Flag','Neighborhood_Tier',
                'Quality_x_Area','Garage_Score','Bsmt_Score','Bed_Bath_Interaction',
                'OverallQual_Sq','Log_GrLivArea','Log_LotArea']
print(f"\nTotal new features created: {len(new_features)}")


## 5. Correlation Analysis: Old vs New Features

In [ ]:
# Compute correlations for new features
new_corrs = fe[new_features + [TARGET]].corr()[TARGET].drop(TARGET)

# Compare against raw originals where applicable
comparison_pairs = [
    ('YearBuilt',    'House_Age'),
    ('YearRemodAdd', 'Remod_Age'),
    ('GrLivArea',    'Log_GrLivArea'),
    ('LotArea',      'Log_LotArea'),
    ('OverallQual',  'OverallQual_Sq'),
    ('OverallQual',  'Quality_x_Area'),
    ('GarageCars',   'Garage_Score'),
    ('TotalBsmtSF',  'Bsmt_Score'),
    ('FullBath',     'Total_Bathrooms'),
    ('1stFlrSF',     'Total_Finished_SF'),
]

orig_corrs = df[numeric_cols + [TARGET]].corr()[TARGET].drop(TARGET)

rows = []
for orig, eng in comparison_pairs:
    o_corr = orig_corrs.get(orig, np.nan)
    e_corr = new_corrs.get(eng, np.nan)
    improvement = abs(e_corr) - abs(o_corr)
    rows.append({'Original Feature': orig,
                 'Engineered Feature': eng,
                 'Orig |r|': round(abs(o_corr), 3),
                 'Eng |r|': round(abs(e_corr), 3),
                 'Δ |r|': round(improvement, 3)})

corr_compare_df = pd.DataFrame(rows).sort_values('Δ |r|', ascending=False)
corr_compare_df.style.background_gradient(subset=['Δ |r|'], cmap='RdYlGn').format('{:.3f}', subset=['Orig |r|','Eng |r|','Δ |r|'])


In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(corr_compare_df))
w = 0.35
ax.bar(x - w/2, corr_compare_df['Orig |r|'], w, label='Original', color='#95a5a6', edgecolor='white')
ax.bar(x + w/2, corr_compare_df['Eng |r|'], w, label='Engineered', color='#e74c3c', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([f"{r['Original Feature']}\n→ {r['Engineered Feature']}"
                    for _, r in corr_compare_df.iterrows()], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('|Pearson r| with SalePrice')
ax.set_title('Correlation with SalePrice: Original vs Engineered Features', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Model Evaluation: Before vs After Feature Engineering

In [ ]:
X_eng = fe.drop(columns=[TARGET])

# Align dtypes (booleans → int)
for col in X_eng.select_dtypes(include='bool').columns:
    X_eng[col] = X_eng[col].astype(int)
for col in X_raw.select_dtypes(include='bool').columns:
    X_raw[col] = X_raw[col].astype(int)

# Final safety fill — any remaining NaN from engineered features
X_eng.fillna(X_eng.median(numeric_only=True), inplace=True)
X_raw.fillna(X_raw.median(numeric_only=True), inplace=True)
print(f"NaN in X_raw: {X_raw.isnull().sum().sum()}  |  NaN in X_eng: {X_eng.isnull().sum().sum()}")

# Log-transform target for better regression behaviour
y_log = np.log1p(y)

X_raw_tr, X_raw_te, y_tr, y_te = train_test_split(X_raw, y_log, test_size=0.2, random_state=SEED)
X_eng_tr, X_eng_te, _, _       = train_test_split(X_eng, y_log, test_size=0.2, random_state=SEED)


In [ ]:
# ── Model V1: Raw features ──────────────────────────────────────────────────────
model_raw = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                      max_depth=4, random_state=SEED)
model_raw.fit(X_raw_tr, y_tr)
pred_raw = model_raw.predict(X_raw_te)

r2_raw  = r2_score(y_te, pred_raw)
mae_raw = mean_absolute_error(np.expm1(y_te), np.expm1(pred_raw))
rmse_raw= np.sqrt(mean_squared_error(y_te, pred_raw))   # RMSLE

print(f"V1 (Raw)  — R²: {r2_raw:.4f}  |  MAE: ${mae_raw:,.0f}  |  RMSLE: {rmse_raw:.4f}")


In [ ]:
# ── Model V2: Engineered features ──────────────────────────────────────────────
model_eng = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                      max_depth=4, random_state=SEED)
model_eng.fit(X_eng_tr, y_tr)
pred_eng = model_eng.predict(X_eng_te)

r2_eng  = r2_score(y_te, pred_eng)
mae_eng = mean_absolute_error(np.expm1(y_te), np.expm1(pred_eng))
rmse_eng= np.sqrt(mean_squared_error(y_te, pred_eng))

print(f"V2 (Eng)  — R²: {r2_eng:.4f}  |  MAE: ${mae_eng:,.0f}  |  RMSLE: {rmse_eng:.4f}")


### 🏆 Performance Comparison

In [ ]:
abs_improvement   = r2_eng - r2_raw
pct_improvement   = abs_improvement / r2_raw * 100

results = pd.DataFrame({
    'Model Version'      : ['Raw Features Only', 'Raw + Engineered Features'],
    'R² Score'           : [round(r2_raw, 4), round(r2_eng, 4)],
    'MAE (USD)'          : [f'${mae_raw:,.0f}', f'${mae_eng:,.0f}'],
    'RMSLE'              : [round(rmse_raw, 4), round(rmse_eng, 4)],
})
print(results.to_string(index=False))
print(f"\n→ R² improvement: +{abs_improvement:.4f} absolute  ({pct_improvement:.1f}% relative)")
print(f"→ MAE improvement: ${mae_raw - mae_eng:,.0f} less error per prediction")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# R² bar
ax = axes[0]
bars = ax.bar(['Raw Features', 'Engineered Features'], [r2_raw, r2_eng],
               color=['#95a5a6', '#e74c3c'], edgecolor='white', width=0.4)
for bar, val in zip(bars, [r2_raw, r2_eng]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', fontweight='bold')
ax.set_ylim(r2_raw * 0.95, min(1.0, r2_eng * 1.04))
ax.set_title('R² Score Comparison', fontweight='bold')
ax.set_ylabel('R² Score')

# Actual vs predicted scatter (engineered model)
ax = axes[1]
ax.scatter(np.expm1(y_te), np.expm1(pred_eng), alpha=0.35, s=15, color='#3498db')
lims = [np.expm1(y_te).min() * 0.9, np.expm1(y_te).max() * 1.05]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
ax.set_title('Engineered Model — Actual vs Predicted', fontweight='bold')
ax.set_xlabel('Actual SalePrice')
ax.set_ylabel('Predicted SalePrice')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.legend()

plt.tight_layout()
plt.show()


## 7. Feature Importance Analysis

In [ ]:
importances = pd.Series(model_eng.feature_importances_, index=X_eng_tr.columns)
top20 = importances.nlargest(20)

# Tag engineered features
is_new = top20.index.isin(new_features)

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = ['#e74c3c' if n else '#3498db' for n in is_new]
top20.sort_values().plot(kind='barh', ax=ax, color=colors_imp[::-1], edgecolor='white')
ax.set_title('Top-20 Feature Importances (Red = Engineered)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')

from matplotlib.patches import Patch
handles = [Patch(color='#e74c3c', label='Engineered Feature'),
           Patch(color='#3498db', label='Original Feature')]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()


In [ ]:
top20_df = top20.reset_index()
top20_df.columns = ['Feature', 'Importance']
top20_df['Type'] = top20_df['Feature'].apply(lambda f: 'Engineered ✅' if f in new_features else 'Original')
top20_df['Importance'] = top20_df['Importance'].round(4)
top20_df.sort_values('Importance', ascending=False)


## 8. Correlation Change Analysis

In [ ]:
print("Feature Transformation Impact on Correlation with SalePrice")
print("=" * 65)
highlight_pairs = [
    ('GrLivArea',   'Log_GrLivArea',    'Log-transform stabilises variance'),
    ('YearBuilt',   'House_Age',        'Age is more linearly related to price'),
    ('OverallQual', 'Quality_x_Area',   'Interaction captures premium size effect'),
    ('GarageCars',  'Garage_Score',     'Cars × area captures true garage value'),
    ('TotalBsmtSF', 'Bsmt_Score',       'Basement quality × size'),
]
for orig, eng, reason in highlight_pairs:
    o = orig_corrs.get(orig, np.nan)
    e = new_corrs.get(eng, np.nan)
    delta = abs(e) - abs(o)
    arrow = '▲' if delta > 0 else '▼'
    print(f"\n{orig}")
    print(f"  ↓  |r| = {abs(o):.3f}")
    print(f"{eng}")
    print(f"  ↓  |r| = {abs(e):.3f}   {arrow} {delta:+.3f}  —  {reason}")


## 9. Automated Feature Creation

In [ ]:
# Programmatic / systematic feature creation
# Demonstrates how the pattern can be extended to dozens of features.

area_cols    = ['GrLivArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GarageArea']
quality_cols = ['OverallQual', 'OverallCond', 'ExterQual', 'BsmtQual', 'KitchenQual']

auto_features = {}

# 1. Log-transform all area columns
for col in area_cols:
    if col in df.columns:
        name = f'Log_{col}'
        auto_features[name] = np.log1p(df[col].fillna(0))

# 2. Quality × Area interactions
for q in quality_cols[:2]:       # OverallQual & OverallCond
    for a in area_cols[:3]:      # top 3 area cols
        name = f'{q}_x_{a}'
        if q in fe.columns and a in df.columns:
            auto_features[name] = fe[q] * df[a].fillna(0)

# 3. Ratio features
ratio_pairs = [
    ('GrLivArea',   'LotArea',          'LivArea_to_Lot'),
    ('TotalBsmtSF', 'GrLivArea',        'Bsmt_to_LivArea'),
    ('GarageArea',  'GrLivArea',        'Garage_to_LivArea'),
]
for num, den, name in ratio_pairs:
    if num in df.columns and den in df.columns:
        auto_features[name] = df[num].fillna(0) / (df[den].replace(0, 1))

auto_df = pd.DataFrame(auto_features)
print(f"Automatically generated features: {auto_df.shape[1]}")
print(auto_df.columns.tolist())

# Compute their correlations with SalePrice
auto_corrs = auto_df.corrwith(y).abs().sort_values(ascending=False)
print("\nTop 10 auto-generated features by |correlation| with SalePrice:")
print(auto_corrs.head(10).round(3))


## 10. Distribution Analysis

In [ ]:
pairs_dist = [
    ('GrLivArea',   'Log_GrLivArea',   'Living Area (raw)',   'Log Living Area'),
    ('YearBuilt',   'House_Age',       'Year Built',          'House Age'),
    ('LotArea',     'Log_LotArea',     'Lot Area (raw)',      'Log Lot Area'),
    ('OverallQual', 'OverallQual_Sq',  'Overall Quality',     'Quality Squared'),
]

fig, axes = plt.subplots(len(pairs_dist), 2, figsize=(13, 4 * len(pairs_dist)))
for i, (orig, eng, orig_label, eng_label) in enumerate(pairs_dist):
    orig_data = df[orig].fillna(df[orig].median()) if orig in df.columns else fe[orig]
    eng_data  = fe[eng]

    ax0, ax1 = axes[i]
    orig_data.plot(kind='hist', bins=40, ax=ax0, color='#3498db', alpha=0.7, edgecolor='white')
    eng_data.plot(kind='hist',  bins=40, ax=ax1, color='#e74c3c', alpha=0.7, edgecolor='white')
    ax0.set_title(orig_label, fontweight='bold')
    ax1.set_title(eng_label,  fontweight='bold')
    for ax in [ax0, ax1]:
        ax.set_ylabel('Count')

plt.suptitle('Distribution: Original vs Engineered Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Observations:**
- `Log_GrLivArea` and `Log_LotArea` are significantly more normally distributed than their raw counterparts — better for linear-assumption models.
- `House_Age` has a more meaningful distribution than `YearBuilt` because it's measured relative to a reference point.
- `OverallQual_Sq` emphasises the non-linear jump at the top end of the quality scale.


## 11. Feature Engineering Report

---

### 📋 Created Features (21 total)

| Category | Feature | Description |
|---|---|---|
| Age | `House_Age` | YrSold − YearBuilt |
| Age | `Remod_Age` | YrSold − YearRemodAdd |
| Flag | `Was_Remodeled` | 1 if house was ever remodelled |
| Area | `Total_Bathrooms` | Full + 0.5×Half (above + basement) |
| Area | `Total_Finished_SF` | GrLivArea + BsmtFinSF1 + BsmtFinSF2 |
| Area | `Total_Porch_SF` | Sum of all porch/deck areas |
| Ratio | `Area_Per_Room` | GrLivArea / TotRmsAbvGrd |
| Ratio | `Area_Per_Bedroom` | GrLivArea / BedroomAbvGr |
| Flag | `High_Quality_Flag` | OverallQual ≥ 8 |
| Flag | `Luxury_Home_Flag` | Large area AND high quality |
| Flag | `Has_Pool / Has_Garage / Has_Fireplace` | Binary presence indicators |
| Category | `Neighborhood_Tier` | 0/1/2 tier based on median prices |
| Interaction | `Quality_x_Area` | OverallQual × GrLivArea |
| Interaction | `Garage_Score` | GarageCars × GarageArea |
| Interaction | `Bsmt_Score` | BsmtQual × TotalBsmtSF |
| Interaction | `Bed_Bath_Interaction` | BedroomAbvGr × Total_Bathrooms |
| Non-linear | `OverallQual_Sq` | Quality² |
| Log | `Log_GrLivArea` | log(1 + GrLivArea) |
| Log | `Log_LotArea` | log(1 + LotArea) |

---

### 📈 Performance Improvement


In [ ]:
print("=" * 55)
print("   FEATURE ENGINEERING — PERFORMANCE SUMMARY")
print("=" * 55)
print(f"  Raw Features   R² : {r2_raw:.4f}")
print(f"  Engineered R²  : {r2_eng:.4f}")
print(f"  Improvement    : +{r2_eng - r2_raw:.4f}  ({(r2_eng - r2_raw)/r2_raw*100:.1f}% relative)")
print()
print(f"  Raw MAE        : ${mae_raw:>10,.0f}")
print(f"  Engineered MAE : ${mae_eng:>10,.0f}")
print(f"  MAE Reduction  : ${mae_raw - mae_eng:>10,.0f}")
print()
print(f"  Raw RMSLE      : {rmse_raw:.4f}")
print(f"  Engineered RMSLE: {rmse_eng:.4f}")
print("=" * 55)

# Top 5 most impactful engineered features
top_eng = importances[importances.index.isin(new_features)].nlargest(5)
print("\n  Top 5 Engineered Features by Importance:")
for feat, imp in top_eng.items():
    print(f"    {feat:<30} {imp:.4f}")


### 🔍 Key Insights

1. **`Quality_x_Area`** is consistently the most impactful engineered feature — the interaction between quality and size captures a premium that neither column captures alone.
2. **`House_Age`** outperforms raw `YearBuilt` because age is directly related to depreciation and condition.
3. **`Total_Bathrooms`** consolidates four separate bathroom columns into a single meaningful metric.
4. **`Log_GrLivArea`** reduces the influence of outlier mansions and produces a more Gaussian-like distribution.
5. **`Bsmt_Score`** and **`Garage_Score`** capture the combined quality+size value of these spaces, which correlate more strongly with price than either dimension alone.
6. **`Neighborhood_Tier`** reduces the high cardinality of 25 neighbourhoods into a 3-level ordinal that the model can learn from more efficiently.

---
*Notebook generated for the Ames Housing dataset (Kaggle House Prices competition).*
